# Why prompt engineering needs evidence: a fictional earnings example

A prompt edit can make an answer look better while quietly making it slower, more expensive, less factual, or impossible to reproduce. This tutorial shows how `aai-core` and MLflow turn that edit into a controlled engineering decision.

You will compare two prompt versions for a **fictional** earnings-summary assistant:

- **v1 — baseline:** summarize only the supplied earnings excerpt;
- **v2 — change:** do the same thing, but include the supplied source identifier exactly once.

Both versions prohibit invented facts and investment recommendations. They run against the same three cases and the same Databricks model, so the citation instruction is the only deliberate change.

By the end, you will understand why teams need prompt versions, traces, runs, experiments, and evaluation gates—not merely how to call their APIs.

> **Safety boundary:** Aster Ridge Systems, every figure, and every source ID in this tutorial are synthetic. Do not put material non-public information, client data, credentials, or personal data into this notebook. This is not investment advice.

## The business problem: a plausible answer is not enough

Imagine a developer changes an earnings-summary prompt and receives one polished response. That response does not answer the questions an engineering team must ask:

- Can someone reproduce exactly which instructions produced it?
- Does the change work across more than one example?
- Did factual coverage or citations improve?
- Did latency, token use, or cost regress?
- Can we debug what happened inside a failed call?

MLflow matters because it preserves evidence for those questions. Without evidence, adopting a prompt is an opinion; with evidence, it becomes a reviewable decision.

```mermaid
flowchart TB
    A[Untracked prompt edit] --> B[One plausible answer]
    B --> C[No reproducible evidence]
    D[Versioned prompt] --> E[Test the same cases]
    E --> F[Compare evidence]
    F --> G[Make a reviewable decision]
```

**Diagram in words:** the top path ends with an answer nobody can reproduce. The bottom path versions the prompt, tests it consistently, and produces evidence for a decision.

## The fictional earnings packet

The same three cases will be sent through both prompt versions as the versioned dataset `fictional-earnings-summary-regression-v1`. Fixed synthetic inputs make the comparison reproducible and prevent current-market claims.

| Case | Question | Approved fictional facts | Source ID |
|---|---|---|---|
| Quarterly performance | What were quarterly revenue and operating margin results? | Revenue `$128.4 million`, up `12%`; operating margin `18.6%`, up from `16.9%` | `ARS-FY25-Q2-RESULTS` |
| Forward guidance | What revenue and operating-margin guidance was provided? | Revenue `$132 million` to `$136 million`; operating margin `19%` to `20%` | `ARS-FY25-Q2-GUIDANCE` |
| Cash flow and risk | What free-cash-flow result and supplier risk were disclosed? | Free cash flow `$21.7 million`; inventory up `28%`; single-source supplier concentration is a risk | `ARS-FY25-Q2-CASH-RISK` |

Why use three cases? A prompt that works for one wording may fail on another. Reusing the exact ordered dataset for v1 and v2 prevents a different input from being mistaken for a prompt improvement.

## MLflow in plain language: a beginner glossary

| Term | Plain-language purpose | Why it matters here |
|---|---|---|
| **Prompt template** | Reusable instructions with placeholders | Separates stable instructions from each earnings case |
| **Prompt Registry** | A version history for prompts—a governed prompt catalog | Makes the exact v1 and v2 text recoverable later |
| **Trace** | The record of what happened during one request | Explains one answer's model call, tokens, cost, and failure state |
| **Span** | One operation inside a trace | Separates the application operation from the provider call |
| **Run** | Evidence collected while testing one version | Groups three comparable calls and their metrics |
| **Experiment** | The long-lived place where runs are compared | Keeps baseline, change, and later decisions together |

A trace answers **“what happened in this call?”** A run answers **“how did this prompt version perform across the test?”** Neither replaces the other.

### Where this evidence lives

MLflow is the evidence system and data model. Databricks can host its managed tracking service and UI. Unity Catalog is the governance layer for selected assets; it is not another name for MLflow.

This notebook deliberately separates execution from evidence. The six LLM requests go to a real Databricks serving endpoint, while the default experiment, runs, traces, and prompt versions stay on this machine. That makes **Run All** safe while someone is learning and avoids requiring workspace artifact permissions.

| Object | This notebook by default | Shared Databricks path |
|---|---|---|
| Experiment and run metadata | `.aai/local/mlflow.db` | [Databricks-hosted MLflow tracking](https://docs.databricks.com/aws/en/mlflow/tracking-server-configuration) in the workspace |
| Traces | Local tracking store, attached to the local experiment | Attached to a workspace experiment; production experiments can be explicitly bound to [Unity Catalog OpenTelemetry tables](https://docs.databricks.com/aws/en/mlflow3/genai/tracing/trace-unity-catalog) |
| Prompt versions | Local Prompt Registry in SQLite | [Unity Catalog Prompt Registry](https://docs.databricks.com/aws/en/mlflow3/genai/prompt-version-mgmt/prompt-registry/create-and-edit-prompts) under `catalog.schema.prompt_name` |
| Evaluation dataset | Fixed in-memory synthetic cases plus a digest | Unity Catalog EvaluationDataset linked as a native input to both runs |
| Run artifacts | `.aai/local/mlruns/...` files | MLflow-managed artifact storage, or a Unity Catalog Volume when configured |
| LLM inference | Databricks serving endpoint | Databricks serving endpoint |
| UI | `make local-ui` | Databricks **Experiments** UI |

The **tracking URI** chooses where experiments, runs, and traces are recorded. The **registry URI** separately chooses where prompt versions are recorded. A three-part name such as `main.example_ai.earnings_summary` is only a reproducible namespace when this tutorial uses SQLite; it becomes a governed Unity Catalog object only with registry URI `databricks-uc`.

Important boundary: publishing prompts to Unity Catalog does **not** automatically move runs or traces there. Unity Catalog trace tables require a separately configured experiment trace location, permissions, and SQL warehouse. Platform operators provision that production option; this tutorial does not create it.

```mermaid
flowchart LR
    A[Finance notebook] -->|request| B[Databricks LLM]
    B -->|response| A
    A -->|records evidence| C[Local MLflow]
    C --> D[Prompts, runs, and traces]
```

**Diagram in words:** this tutorial sends model requests to Databricks, but records prompts, runs, and traces in local MLflow. The table above explains the optional shared Databricks storage choices.

## 1. Prepare the environment

**Why this matters:** installing packages in one Python environment while the notebook uses another is a common source of confusing import failures.

**What risk it prevents:** accidentally running unpinned or incomplete dependencies.

**What evidence we will collect:** the active kernel and exact tracking/registry destinations selected for this run.

From the repository root, run:

```bash
make examples-install
az login
cp aai-platform.example.yml aai-platform.yml  # only when absent
```

Then select `<repository>/.venv/bin/python` as the notebook kernel. In VS Code use **Select Kernel → Python Environments**. You can run [`05_connected_setup.ipynb`](05_connected_setup.ipynb) first to diagnose these two checkpoints in isolation; this tutorial calls the same shared setup functions, so it remains safe to use **Run All** directly.

The next cell has one switch: `SEND_EVIDENCE_TO_DATABRICKS`. Keep it `False` for local MLflow at `.aai/local/mlflow.db`, or set it to `True` to send the experiment, runs, traces, and exact prompt versions to Databricks-hosted MLflow and the Unity Catalog Prompt Registry. Restart the kernel after changing the switch because MLflow tracing is configured once per process.

| Switch | Tracking URI | Registry URI | What is stored remotely |
|---|---|---|---|
| `False` | Local SQLite | Local SQLite | Only the six LLM inference requests leave the machine |
| `True` | `databricks` | `databricks-uc` | Experiment, UC dataset, runs, traces, prompt versions, metrics, and lineage links |

| Failure | Meaning | Action |
|---|---|---|
| `.venv` is not offered as a kernel | Notebook support is not installed in that environment | Run `make examples-install`, reload VS Code, then select `.venv/bin/python` |
| Missing Python module | Wrong kernel or examples not installed | Run `make examples-install`, select `.venv/bin/python`, restart kernel |
| Missing configuration | No local environment mapping | Copy `aai-platform.example.yml` and configure it |
| Wrong Azure tenant | Login belongs to another organization | Run the tenant-specific command printed by the preflight |
| Endpoint placeholder | No model deployment selected | Choose one of the visible `READY` chat endpoints |
| 403 on inference | Identity lacks endpoint permission | Request least-privilege `CAN_QUERY` |


In [ ]:
import importlib
import sys
from pathlib import Path

# VS Code may start the kernel at the repository root or inside examples/.
repo_root = next(
    (
        directory
        for directory in (Path.cwd(), *Path.cwd().parents)
        if (directory / "examples" / "notebook_setup.py").is_file()
    ),
    None,
)
if repo_root is None:
    raise FileNotFoundError(
        "Open the cloned repository as your VS Code workspace, then restart "
        "the notebook kernel."
    )
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

# False is the safe default. Set True, restart the kernel, then Run All to
# store prompts, runs, traces, metrics, and lineage in Databricks.
SEND_EVIDENCE_TO_DATABRICKS = False

# The helper keeps dependency, auth, and MLflow routing out of the lesson.
setup_helpers = importlib.import_module("examples.notebook_setup")
environment = setup_helpers.prepare_notebook_environment(
    repo_root,
    evidence_destination=("databricks" if SEND_EVIDENCE_TO_DATABRICKS else "local"),
)
config_path = environment.config_path
mlflow = environment.mlflow

### Interpret the setup checkpoint

`SETUP PASSED` means this kernel can import the required packages, found the repository configuration, and configured the printed evidence destination. Local mode uses SQLite plus `.aai/local/mlruns`; Databricks mode uses tracking URI `databricks` and registry URI `databricks-uc`. It does **not** prove that Azure authentication or endpoint access works; that is the next, separate checkpoint.

## 2. Verify identity, workspace, and endpoint

**Why this matters:** authentication proves who you are; authorization determines what that identity may query. Checking both before prompt work makes later failures specific instead of mysterious.

**What risk it prevents:** discovering identity or endpoint problems only after an experiment has started.

**What evidence we will collect:** tenant, workspace identity, endpoint type, and readiness.

A Databricks CLI profile is not required. The notebook uses the repository's non-secret workspace host and Databricks unified `azure-cli` authentication. The detailed checks live in `examples/notebook_setup.py`, which is also used by the dedicated setup notebook.

In [ ]:
# Fail on identity or endpoint access before registering prompts or making calls.
connected = setup_helpers.preflight_databricks(environment)

# The tutorial uses these named results; diagnostic plumbing stays in the helper.
ctx = connected.context
model = connected.model
experiment_name = connected.experiment_name

### Interpret the cloud-access checkpoint

`PREFLIGHT PASSED` means the Azure tenant, workspace identity, endpoint type, and endpoint readiness agree with the configuration. Reading endpoint metadata still does not prove `CAN_QUERY`; the six real calls provide that evidence.

## 3. Register the prompts before invoking them

**Why this matters:** a raw string in a notebook can be edited without leaving evidence. A registered prompt version is immutable, addressable, and comparable.

**What risk it prevents:** without an exact version URI and digest, a future developer cannot prove which instructions produced a result.

**What evidence we will collect:** two immutable prompt URIs and content digests.

The prompts intentionally differ in only one behavior: v2 must cite the supplied source identifier. The v2-only source field carries the value needed to satisfy that requirement; every summarization and recommendation-policy instruction stays unchanged.

| v1 — evidence-only baseline | v2 — cited change |
|---|---|
| Shared earnings-summary instructions<br>Question + fictional excerpt | Same shared instructions<br>**Add:** include the source identifier exactly once<br>Question + same fictional excerpt<br>**Add:** source identifier value |

Here is the exact text delta:

```diff
 Summarize the fictional earnings excerpt using only the
 supplied facts.
 Aster Ridge Systems and every figure in this exercise are fictional.
 Do not provide investment advice or recommend buying, selling, or holding securities.
+Include the supplied source identifier exactly once.
 
 Question: {{question}}
 Fictional earnings excerpt: {{earnings_excerpt}}
+Source identifier: {{source_id}}
```

Isolating the citation behavior lets the comparison attribute a citation change to that requirement rather than to unrelated prompt wording.

### Exact versions versus aliases

An exact version and an alias solve different problems:

| Reference | Behavior | Correct use |
|---|---|---|
| Exact URI such as `prompts:/.../7` | Immutable | Evaluation evidence, trace/run lineage, rollback, and reproduction |
| Alias such as `@development` | Movable pointer | The version currently being edited or exercised |
| Alias such as `@validation` | Movable pointer | The exact version submitted to a release-grade gate |
| Alias such as `@production` | Movable pointer | The version an approved production deployment should resolve |

The SDK owns the allowed alias vocabulary and the registry operation through `PromptManager.set_alias()`. The experiment or release workflow owns **when** an alias may move. This exploratory notebook moves no alias because its conclusion is always inconclusive. The deterministic `04_first_evaluation.py` stage moves `production` to v2 only after every gate passes. Even then, its evidence continues to record the exact immutable version.

```mermaid
flowchart LR
    A[Same cases and model] --> B[Prompt v1]
    A --> C[Prompt v2]
    B --> D[Compare results]
    C --> D
```

**Diagram in words:** both prompts receive the same cases and model settings. The citation instruction is the only intended difference.

In [ ]:
import importlib

from examples import lifecycle_support
from examples.support.connected_llm import prepare_prompt_pair, prompt_pair_summary

importlib.reload(lifecycle_support)
CASES = lifecycle_support.CASES
DATASET_NAME = lifecycle_support.DATASET_NAME
PROMPT_NAME = lifecycle_support.PROMPT_NAME
dataset_digest = lifecycle_support.dataset_digest
prompt_digest = lifecycle_support.prompt_digest

experiment_name = lifecycle_support.prepare_mlflow(ctx)
prompt_pair = prepare_prompt_pair(ctx)  # Registration is idempotent.
prompts, prompt_versions = prompt_pair.manager, prompt_pair.versions
loaded_prompts = prompt_pair.loaded  # Render the registry-loaded object.
baseline_prompt_version, change_prompt_version = (
    prompt_versions["baseline"],
    prompt_versions["change"],
)
loaded_baseline_prompt, loaded_change_prompt = (
    loaded_prompts["baseline"],
    loaded_prompts["change"],
)
print("PROMPT VERSIONS READY")
print(prompt_pair_summary(prompt_pair))

### Interpret the prompt checkpoint

`PROMPT VERSIONS READY` proves that both prompt contents are stored and loaded by immutable version. The returned version numbers may not literally be `1` and `2` after earlier runs; the printed exact URIs and digests—not a guessed number or mutable alias—are the reproducibility evidence.

## 4. Run the controlled A/B test

**Why this matters:** prompt engineering is experimental work. Both versions must see the same inputs, model, temperature, and token limit or the comparison is confounded.

**What risk it prevents:** mistaking a dataset, model, or sampling difference for a prompt improvement.

**What evidence we collect:** each version gets one MLflow run containing three real calls. Each call gets its own trace linked to the exact prompt version that rendered it.

`aai-core` resolves the logical model name `general-chat`. `create_native_async_client()` returns the recognizable provider client with the same governed identity, gateway, timeout, and retry configuration. This notebook owns one client for its event loop and closes it with `async with`. MLflow's OpenAI integration owns the provider span. The same call must not also use `model.generate()`, because that would create a duplicate provider span and block the active event loop.

In [ ]:
from examples.support.connected_llm import (
    configure_comparison_tracing,
    register_evaluation_dataset,
)

configure_comparison_tracing(ctx, experiment_name)
registered_dataset = register_evaluation_dataset(
    environment=environment,
    connected=connected,
    setup_helpers=setup_helpers,
    mlflow_module=mlflow,
)
if registered_dataset is not None:
    print(
        {
            "unity_catalog_dataset": registered_dataset.name,
            "dataset_id": registered_dataset.dataset_id,
            "experiment_id": connected.experiment_id,
        }
    )

In [ ]:
from examples.support.connected_llm import run_prompt_comparison

call_records, run_ids, client = await run_prompt_comparison(
    ctx=ctx,
    model=model,
    experiment_name=experiment_name,
    pair=prompt_pair,
    registered_dataset=registered_dataset,
    mlflow_module=mlflow,
)
if len(call_records) != len(CASES) * 2:
    raise RuntimeError(f"Expected two calls per case; recorded {len(call_records)}.")
print("A/B CALLS SUCCEEDED")
print(
    {
        "calls": len(call_records),
        "cases_per_prompt": len(CASES),
        "baseline_run_id": run_ids["baseline"],
        "change_run_id": run_ids["change"],
    }
)

### Interpret the A/B checkpoint

`A/B CALLS SUCCEEDED` proves that all six intended endpoint calls completed: three identical cases through each exact prompt version. It proves execution, not superiority; the responses still need trace verification and scoring.

## 5. Read traces like debugging evidence

**Why this matters:** an aggregate score can say that something failed, but a trace helps explain where and how one request behaved.

**What risk it prevents:** hiding a broken or duplicated provider operation inside a reassuring average.

**What evidence we will collect:** span shape, SDK metadata, provider usage, cost coverage, and per-case scores.

This tutorial expects each trace to contain one application span and one provider span. Token usage and cost belong to the provider operation; SDK-controlled application metadata belongs to the whole trace.

The score meanings are deliberately simple:

- **Fact coverage:** the fraction of required fictional facts present; presentation-only Markdown and typography do not count as factual differences.
- **Exact citation:** `1` only when the literal source ID occurs exactly once.
- **Recommendation compliance:** `1` when no buy, sell, or hold recommendation appears.
- **Quality score:** the mean of fact coverage and exact citation; recommendation compliance remains a separate policy check.
- **Cost coverage:** the fraction of calls for which the provider supplied cost evidence. Unknown cost is never treated as zero.

```mermaid
flowchart LR
    A[One trace] --> B[Application span]
    B --> C[Model-call span]
    D[Exact prompt version] --> A
    E[Comparison run] --> A
```

**Diagram in words:** one trace contains the application operation and its model call. Links identify the exact prompt and the comparison run.

In [ ]:
from IPython.display import display

from examples.support.connected_llm import (
    TRACE_DISPLAY_COLUMNS,
    build_trace_evaluation,
)

evaluation_frame = build_trace_evaluation(
    call_records=call_records,
    run_ids=run_ids,
    environment=environment,
    experiment_name=experiment_name,
    ctx=ctx,
    client=client,
    mlflow_module=mlflow,
)
print("TRACES VERIFIED")
display(evaluation_frame[list(TRACE_DISPLAY_COLUMNS)])

### Interpret the trace checkpoint

`TRACES VERIFIED` means every call has SDK metadata, the expected two-span shape, its source-run association, and exact prompt-version lineage. In local mode the notebook reads both spans directly. In Databricks mode it reads trace metadata and run lineage without downloading the full span artifact; inspect the complete span payload in the Databricks experiment UI. This avoids a workstation-side signed-blob `403` while still proving the trace was persisted remotely.

Fact coverage ignores presentation-only Markdown, spacing, and typographic hyphens; exact citation deliberately does not, because changing even one source-ID character violates that contract. Missing cost remains missing evidence, never an assumed zero.

## 6. Compare trade-offs before deciding

**Why this matters:** the “best” prompt is not merely the one with the nicest answer. A production change must meet quality requirements without unacceptable latency, token, or cost regressions.

**What risk it prevents:** promoting a quality improvement whose operational cost or reliability is unacceptable.

**What evidence we will collect:** aggregated quality, policy, latency, token, cost, and coverage metrics for both versions.

This notebook records the comparison but deliberately returns **inconclusive**. Three cases with one stochastic repetition are useful development evidence, not enough evidence for release.

```mermaid
flowchart LR
    A[Comparison results] --> B{Enough evidence?}
    B -->|Not yet| C[Run full evaluation]
    B -->|Pass| D[Adopt]
    B -->|Fail| E[Reject]
```

**Diagram in words:** incomplete evidence leads to the full evaluation. Only release-grade evidence can lead to adopt or reject.

In [ ]:
from examples.support.connected_llm import record_exploratory_comparison

comparison, decision_record = record_exploratory_comparison(
    evaluation_frame, run_ids=run_ids, client=client
)
# Six exploratory calls cannot authorize release.
decision, next_action = decision_record["decision"], decision_record["next_action"]
print("COMPARISON RECORDED")
display(comparison)
print(
    {
        **decision_record,
        "experiment": experiment_name,
        "baseline_prompt_uri": baseline_prompt_version.uri,
        "change_prompt_uri": change_prompt_version.uri,
    }
)

### Interpret the comparison checkpoint

`COMPARISON RECORDED` means the two runs now carry comparable aggregate evidence and an explicit decision. `observed_preference` describes only this small live sample. The decision remains `inconclusive`, the release remains blocked, and the next action is the deterministic full evaluation.

## 7. Inspect the evidence in MLflow

**Why this matters:** notebook output disappears from view; MLflow preserves the comparison so another developer can review the exact prompts, runs, traces, metrics, and tags.

**What risk it prevents:** making a decision from evidence that only exists in one developer's scrolling notebook output.

**What evidence we will collect:** durable runs, trace IDs, prompt links, metrics, and decision tags in the selected evidence store.

When `SEND_EVIDENCE_TO_DATABRICKS = False`, run `make local-ui` in another terminal:

```bash
make local-ui
```

Open `http://127.0.0.1:5000` and select `/Shared/example-ai-earnings-summary-quality-cost`. When the switch is `True`, open the configured Databricks workspace, go to **Experiments**, and select the same experiment name. The printed run and trace IDs are identical to the remote records.

Look for:

1. Run `connected-baseline-earnings-summary-prompt-v1` and run `connected-change-cited-earnings-summary-prompt-v2`.
2. Three traces associated with each run.
3. Exact prompt-version links—not only names or aliases.
4. Citation improvement alongside latency, token, and cost evidence.
5. The `inconclusive` decision and `run_full_evaluation` next action.

## 8. Optional: publish only the prompts from local mode

**Why this matters:** the local registry makes learning reliable; a governed Databricks/Unity Catalog registry makes prompt versions discoverable and permission-controlled across a team.

**What risk it prevents:** treating a developer-local prompt record as if it were shared production governance.

**What evidence we will collect:** remote immutable prompt URIs and matching content digests, without invalid cross-store links.

This section is only for a local comparison whose prompts should also be copied to Unity Catalog. Setting the guard to `True` creates or reuses prompt versions in the configured Unity Catalog Prompt Registry and therefore requires catalog/schema permissions. It changes only the registry URI; tracking remains local.

Local traces are **not** linked to these copied remote prompt versions. Lineage links must stay within one backend. For complete remote lineage, leave this guard `False`, set `SEND_EVIDENCE_TO_DATABRICKS = True` at the top, restart the kernel, and use **Run All**. That mode registers the prompts remotely before inference and links every remote trace to its exact remote prompt and run.

In [ ]:
from examples.support.connected_llm import publish_prompt_pair_to_databricks

# Only prompt storage changes here; local traces are never linked cross-store.
PUBLISH_PROMPTS_TO_DATABRICKS = False
if environment.evidence_destination.value == "databricks":
    print("FULL DATABRICKS EVIDENCE MODE ALREADY ACTIVE")
elif not PUBLISH_PROMPTS_TO_DATABRICKS:
    print("OPTIONAL DATABRICKS SECTION SKIPPED")
else:
    published_prompts = publish_prompt_pair_to_databricks(ctx=ctx, mlflow_module=mlflow)
    print("DATABRICKS PROMPT VERSIONS READY")
    print(published_prompts)

### Interpret the optional publishing result

When disabled, the skip message confirms that **Run All** made no remote Prompt Registry changes. When enabled, the printed remote URIs and verified digests prove publication only; they do not create lineage to the local traces or runs.

## What this proved—and what it did not

### You proved

- `aai-core` can resolve a logical model and call a real Databricks LLM.
- MLflow can preserve immutable prompt versions before they are invoked.
- Six calls can be traced and linked to the exact prompt and comparison run.
- The same cases can expose quality, citation, latency, token, and cost trade-offs.
- Evidence can support an explicit decision instead of an undocumented prompt edit.

### You did not prove

- That the output is financial advice or suitable for an investment decision.
- That three synthetic cases represent production traffic.
- That one stochastic repetition is enough for release.
- That the configured model, prompt, or endpoint is approved for production.

Run `make local-example EXAMPLE=first_evaluation` for the deterministic full gate that compares exact versions across the shared regression dataset.

## Troubleshooting

- **Start clean:** restart the kernel and use **Run All** after dependency or configuration changes.
- **Kernel unavailable in VS Code:** run `make examples-install`, reload the VS Code window, then select `<repository>/.venv/bin/python`.
- **Prompt already exists:** expected; registration reuses identical content by digest rather than creating duplicates.
- **No token usage:** the endpoint did not return an OpenAI-compatible usage object; tracing succeeded, but cost evidence is incomplete.
- **No cost:** cost is unknown, not zero. The notebook records incomplete cost coverage.
- **Local database locked:** stop another `make local-ui` process before retrying writes.
- **Mermaid not rendered:** use the prose immediately below each diagram; no execution depends on Mermaid.
- **Databricks CLI profile:** this notebook explicitly selects the configured host and `azure-cli` authentication, so a profile is neither required nor automatically selected. Restart the kernel after changing authentication-related environment variables.
- **Databricks endpoint or registry 403/404:** rerun preflight and verify workspace membership, endpoint name, `CAN_QUERY`, and catalog/schema privileges.
- **Signed-blob 403 while opening a remote trace locally:** the trace can still be stored successfully. This notebook verifies it through metadata-only search; inspect the full spans in the Databricks experiment UI. Ask the workspace owner to diagnose artifact-storage access only if your workflow requires downloading span payloads to the workstation.
- **Optional registry permission error:** leave publishing disabled or request least-privilege catalog/schema Prompt Registry access.